In [ ]:
import json
import tiktoken

#import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma

from dotenv.ipython import load_dotenv
import os

c:\Users\AdMin\OneDrive\Desktop\School\4IIR\S2\SystemMultiAgent\tpG\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
load_dotenv(override=True)

True

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [ ]:
pdf_file = "./pdfs/Rapport Financier Annuel  OCP 2023.pdf"

In [ ]:
pdf_loader = PyPDFLoader(pdf_file)

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader(path = "./pdfs")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='o200k_base',
    chunk_size=300,
    chunk_overlap=20
)

In [ ]:
chunks = loader.load_and_split(text_splitter)

In [ ]:
len(chunks)

576

In [ ]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model='text-embedding-ada-002')

In [ ]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding_model,
    collection_name="rapport_ocp_V2",
    persist_directory="./store"
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [ ]:
retrieved_chunks=retriever.invoke("Quelles sont les performances financières de l'OCP en 2023?")

In [ ]:
print(retrieved_chunks)

[Document(metadata={'page_label': '136', 'creationdate': '2024-04-29T21:13:52+01:00', 'creator': 'Nitro Pro 13 (13.32.0.623)', 'producer': 'PyPDF', 'page': 135, 'total_pages': 210, 'moddate': '2024-04-30T08:36:20+01:00', 'source': 'pdfs\\Rapport Financier Annuel  OCP 2023.pdf'}, page_content='RÉSULTATS DE L’EXERCICE 2023 \nD’OCP S.A'), Document(metadata={'page': 126, 'moddate': '2024-04-30T08:36:20+01:00', 'page_label': '127', 'creationdate': '2024-04-29T21:13:52+01:00', 'source': 'pdfs\\Rapport Financier Annuel  OCP 2023.pdf', 'creator': 'Nitro Pro 13 (13.32.0.623)', 'producer': 'PyPDF', 'total_pages': 210}, page_content='1. Contexte de l’activité 03\n2. Activité durant l’année 2023 07\n3. Résultats de l’exercice 2023 d’OCP S.A 11'), Document(metadata={'page_label': '3', 'creator': 'Nitro Pro 13 (13.32.0.623)', 'producer': 'PyPDF', 'creationdate': '2024-04-29T21:13:52+01:00', 'moddate': '2024-04-30T08:36:20+01:00', 'page': 2, 'total_pages': 210, 'source': 'pdfs\\Rapport Financier Annu

In [ ]:
print(len(retrieved_chunks))

5


In [ ]:
prompt_template = """
Answer the following question based only on provided context
The context is about OCP Annual Financial Report 2023
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer : JE NE SAIS PAS
<context>
 {context}
</context>
<question>
 {question}
</question>
"""

In [ ]:
user_input = "Quelles sont les performances financières de l'OCP en 2023?"

In [ ]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [ ]:
context_for_query

"RÉSULTATS DE L’EXERCICE 2023 \nD’OCP S.A. 1. Contexte de l’activité 03\n2. Activité durant l’année 2023 07\n3. Résultats de l’exercice 2023 d’OCP S.A 11. Chiffres clés\nLes résultats financiers d’OCP pour les années 2022 et 2023 sont synthétisés dans les indicateurs clés présentés \nci-dessus.\nEn 2023, les performances financières de OCP ont maintenu une certaine stabilité, témoignant de la résilience du \nGroupe face à des conditions de marché fluctuantes. En effet, les fondamentaux du Groupe sont demeurés solides, \nce qui lui a permis de maintenir une position compétitive sur le marché mondial des engrais.\nIl convient de souligner que l’année 2022 a été exceptionnelle pour le Groupe OCP, cette période a été caractérisée \npar une augmentation significative du chiffre d’affaires, soutenue par la hausse des prix des produits tels que \nl’acide phosphorique et les engrais sur le marché mondial, dans ce qui pourrait être qualifié d’une année de forte \naugmentation généralisée des pr

In [ ]:
# Here the length is 10 because, earlier we have declared k=10
len(relevant_document_chunks)

5

In [ ]:
prompt = prompt_template.format(context=context_for_query, question=user_input)

In [ ]:
print(prompt)


Answer the following question based only on provided context
The context is about OCP Annual Financial Report 2023
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer : JE NE SAIS PAS
<context>
 RÉSULTATS DE L’EXERCICE 2023 
D’OCP S.A. 1. Contexte de l’activité 03
2. Activité durant l’année 2023 07
3. Résultats de l’exercice 2023 d’OCP S.A 11. Chiffres clés
Les résultats financiers d’OCP pour les années 2022 et 2023 sont synthétisés dans les indicateurs clés présentés 
ci-dessus.
En 2023, les performances financières de OCP ont maintenu une certaine stabilité, témoignant de la résilience du 
Groupe face à des conditions de marché fluctuantes. En effet, les fondamentaux du Groupe sont demeurés solides, 
ce qui lui a permis de maintenir une position compétitive sur le marché mondial des engrais.
Il convient de souligner que l’année 2022 a été exceptionnelle pour le Groupe OCP, cette période a été ca

In [ ]:
resp = llm.invoke(prompt)

In [ ]:
from IPython.display import Markdown

In [ ]:
print(display(Markdown(resp.content)))

En 2023, les performances financières de l'OCP ont maintenu une certaine stabilité, témoignant de la résilience du Groupe face à des conditions de marché fluctuantes. Les fondamentaux du Groupe sont demeurés solides, ce qui lui a permis de maintenir une position compétitive sur le marché mondial des engrais.

None


In [ ]:
def RAG(query, llm=llm, prompt_template=prompt_template):
    context_docs = retriever.invoke(query)
    context_list = [d.page_content for d in context_docs]
    context_for_query = ". ".join(context_list)
    prompt = prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

In [ ]:
response = RAG("État du résultat global consolidé")
print(display(Markdown(response)))

Le résultat global consolidé pour l'exercice 2023 est de 14 187 millions de dirhams, tandis que pour l'exercice 2022, il était de 27 629 millions de dirhams.

None


In [ ]:
user_input = "j'ai fain et je veux manger quelque chose"
output = RAG(user_input)
print(output)

JE NE SAIS PAS


In [ ]:
response = RAG("Chiffre d'affaire de l'OCP en 2023")
print(display(Markdown(response)))

Le chiffre d'affaires de l'OCP en 2023 est de 81 239 323 844 Dirhams.

None


In [ ]:
response = RAG("Quelles sont les performances financières de l'OCP en 2023")
print(display(Markdown(response)))

En 2023, les performances financières de l'OCP ont maintenu une certaine stabilité, témoignant de la résilience du Groupe face à des conditions de marché fluctuantes. Les fondamentaux du Groupe sont demeurés solides, ce qui lui a permis de maintenir une position compétitive sur le marché mondial des engrais.

None


In [ ]:
user_input ="État du résultat global consolidé"

In [ ]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [ ]:
user_message_template = """
 ###Question
 {question}
 ###Context
 {context}
 ###Answer
 {answer}
"""

In [ ]:
# Default answer for an RAG query
answer = RAG(user_input)
print(display(Markdown(answer)))

Le résultat global consolidé pour l'exercice 2023 est de 14 187 millions de dirhams, tandis que pour l'exercice 2022, il était de 27 629 millions de dirhams.

None


In [ ]:
groundedness_rater_system_message="""
Vous êtes chargé d’évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d’IA pour générer la réponse, ainsi qu’une réponse générée par l’IA à la question.

Dans l’entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par l’IA commencera par ###Answer.

Critères d’évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1 — La métrique n’est pas respectée du tout
2 — La métrique n’est respectée que dans une mesure limitée
3 — La métrique est respectée dans une bonne mesure
4 — La métrique est respectée en grande partie
5 — La métrique est entièrement respectée

Métrique :
La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions :

Écrivez d’abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d’évaluation et attribuer un score.
"""

In [ ]:
groundness_checker = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

In [ ]:
def evaluate(system_message,user_message_template, question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list = [d.page_content for d in retrieved_chunks]
    context = ". ".join(context_list)
    answer = RAG(question)
    prompt = f"""
     {system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
    """
    juge_response= model.invoke(prompt)
    return juge_response.content


In [ ]:
resp=evaluate(groundedness_rater_system_message, user_message_template, user_input)

In [ ]:
print(display(Markdown(resp)))

Pour évaluer la réponse selon la métrique, voici les étapes nécessaires :

1. **Identifier la question** : La question demande des informations sur l'état du résultat global consolidé.

2. **Analyser le contexte** : Le contexte fournit des données financières consolidées pour le Groupe OCP, y compris le résultat global consolidé pour les exercices 2023 et 2022.

3. **Vérifier la réponse par rapport au contexte** : La réponse indique que le résultat global consolidé pour l'exercice 2023 est de 14 187 millions de dirhams et pour l'exercice 2022, il est de 27 629 millions de dirhams.

4. **Comparer les informations de la réponse avec celles du contexte** : 
   - Dans le contexte, il est mentionné que le "Résultat global consolidé" pour l'exercice 2023 est de 14 187 millions de dirhams.
   - Pour l'exercice 2022, le "Résultat global consolidé" est de 27 629 millions de dirhams.

5. **Évaluer la conformité de la réponse avec la métrique** : La réponse est entièrement dérivée des informations fournies dans le contexte. Les chiffres mentionnés dans la réponse correspondent exactement à ceux du contexte.

**Explication étape par étape** :

- La question demande des informations spécifiques sur le résultat global consolidé.
- Le contexte fournit ces informations de manière claire et précise.
- La réponse reprend exactement les chiffres du contexte sans ajouter d'informations externes ou incorrectes.
- Il n'y a pas de divergence entre les données du contexte et celles de la réponse.

**Évaluation** :

La réponse respecte entièrement la métrique, car elle est directement dérivée des informations fournies dans le contexte sans aucune déviation. Par conséquent, la réponse mérite un score de 5.

None
